# CS6735 Data Mining — Final Project
## Feature Selection: Relief / ReliefF

**Project:** FS6 — Relief / ReliefF  
**Objective:** Evaluate how ReliefF feature selection affects classification performance across multiple datasets.  
**Experimental design:** `(#datasets) × 2 phases × 5 classifiers`, 10-fold stratified cross-validation.

---

## 0. Install dependencies
Run this cell once if `skrebate` is not installed.

In [1]:
# !pip install skrebate scikit-learn pandas numpy matplotlib seaborn

## 1. Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from skrebate import ReliefF

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

print('All imports successful.')

All imports successful.


## 2. Configuration — edit this cell for your setup

In [3]:
# ── Dataset paths ────────────────────────────────────────────────────────────
# Add as many datasets as provided. Each entry: 'name': 'path/to/train.csv'
# If you have separate test CSVs, add a second path; otherwise leave as None.
DATASETS = {
    'Dataset1' : {'train': 'Datasets/1/train.csv','test': None},
    'Dataset2' : {'train': 'Datasets/2/train.csv','test': None},
    'Dataset3' : {'train': 'Datasets/3/train.csv','test': None},
    'Dataset4' : {'train': 'Datasets/4/train.csv','test': None},
    'Dataset5' : {'train': 'Datasets/5/train.csv','test': None},
    'Dataset6' : {'train': 'Datasets/6/train.csv','test': None},
    'Dataset7' : {'train': 'Datasets/7/train.csv','test': None},
    'Dataset8' : {'train': 'Datasets/8/train.csv','test': None},
    'Dataset9' : {'train': 'Datasets/9/train.csv','test': None},
    'Dataset10' : {'train': 'Datasets/10/train.csv','test': None},
    'Dataset11' : {'train': 'Datasets/11/train.csv','test': None},
    'Dataset12' : {'train': 'Datasets/12/train.csv','test': None},
    'Dataset13' : {'train': 'Datasets/13/train.csv','test': None},
    'Dataset14' : {'train': 'Datasets/14/train.csv','test': None},
    'Dataset15' : {'train': 'Datasets/15/train.csv','test': None},
    'Dataset16' : {'train': 'Datasets/16/train.csv','test': None},
}

LABEL_COL   = 'Label'   # column name for the target label
N_FOLDS     = 10        # outer cross-validation folds
INNER_FOLDS = 5         # inner CV folds for hyperparameter tuning
RANDOM_STATE = 42

# ── ReliefF settings ─────────────────────────────────────────────────────────
# k = number of top features to keep after ranking
# Try a few values; the best k is selected by inner CV
K_CANDIDATES = [10, 20, 50, 100]  # adjust based on total feature count
RELIEFF_N_NEIGHBORS = 10          # ReliefF neighbourhood size

print('Configuration loaded.')

Configuration loaded.


## 3. Classifier definitions with hyperparameter grids

In [4]:
CLASSIFIERS = {
    'SVM': (
        SVC(random_state=RANDOM_STATE),
        {
            'classifier__C':      [0.1, 1, 10],
            'classifier__kernel': ['rbf', 'linear'],
        }
    ),
    'kNN': (
        KNeighborsClassifier(),
        {
            'classifier__n_neighbors': [3, 5, 11, 21],
            'classifier__weights':     ['uniform', 'distance'],
        }
    ),
    'Decision Tree': (
        DecisionTreeClassifier(random_state=RANDOM_STATE),
        {
            'classifier__max_depth':        [5, 10, 20, None],
            'classifier__min_samples_split': [2, 5, 10],
        }
    ),
    'Random Forest': (
        RandomForestClassifier(random_state=RANDOM_STATE),
        {
            'classifier__n_estimators': [50, 100, 200],
            'classifier__max_depth':    [5, 10, None],
        }
    ),
    'MLP': (
        MLPClassifier(max_iter=500, random_state=RANDOM_STATE),
        {
            'classifier__hidden_layer_sizes': [(64,), (128,), (128, 64)],
            'classifier__alpha':              [1e-4, 1e-3, 1e-2],
        }
    ),
}

print(f'{len(CLASSIFIERS)} classifiers defined:', list(CLASSIFIERS.keys()))

5 classifiers defined: ['SVM', 'kNN', 'Decision Tree', 'Random Forest', 'MLP']


## 4. Helper functions

In [5]:
def load_dataset(path, label_col=LABEL_COL):
    """Load a CSV, separate features and labels, encode string labels."""
    df = pd.read_csv(path)
    X = df.drop(columns=[label_col]).values.astype(np.float64)
    y_raw = df[label_col].values
    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    print(f'  Loaded: {X.shape[0]} samples, {X.shape[1]} features, '
          f'{len(np.unique(y))} classes')
    return X, y, le


def apply_relieff(X_train, y_train, X_test, k, n_neighbors=RELIEFF_N_NEIGHBORS):
    """Fit ReliefF on training data only, transform both train and test."""
    selector = ReliefF(n_features_to_select=k, n_neighbors=n_neighbors)
    selector.fit(X_train, y_train)
    top_idx = np.argsort(selector.feature_importances_)[::-1][:k]
    return X_train[:, top_idx], X_test[:, top_idx], selector.feature_importances_, top_idx


def build_pipeline(clf):
    """Wrap a classifier in imputer + scaler pipeline."""
    return Pipeline([
        ('imputer',    SimpleImputer(strategy='mean')),
        ('scaler',     StandardScaler()),
        ('classifier', clf),
    ])


def evaluate_classifier(clf, param_grid, X_train, y_train, X_test, y_test):
    """Run inner GridSearchCV and return accuracy and macro-F1 on the test fold."""
    pipe = build_pipeline(clf)
    inner_cv = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True,
                                random_state=RANDOM_STATE)
    gs = GridSearchCV(pipe, param_grid, cv=inner_cv,
                      scoring='f1_macro', n_jobs=-1, refit=True)
    gs.fit(X_train, y_train)
    y_pred = gs.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='macro', zero_division=0)
    best_params = gs.best_params_
    return acc, f1, best_params


print('Helper functions defined.')

Helper functions defined.


In [ ]:
import time

# Just use dataset 16 for the test
X, y, le = load_dataset('Datasets/16/train.csv')

# Use a single simple split instead of full CV
split = int(len(X) * 0.9)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f'Training size: {X_train.shape}')
print(f'Test size: {X_test.shape}\n')

for clf_name, (clf, grid) in CLASSIFIERS.items():
    print(f'Testing {clf_name}...', end=' ')
    start = time.time()
    acc, f1, params = evaluate_classifier(clf, grid, X_train, y_train, X_test, y_test)
    elapsed = time.time() - start
    print(f'{elapsed:.1f}s — acc={acc:.3f} f1={f1:.3f}')
```

This will print something like:
```
Testing SVM...        45.2s — acc=0.912 f1=0.891   ← this is your culprit
Testing kNN...         3.1s — acc=0.887 f1=0.865
Testing Decision Tree  1.2s — acc=0.834 f1=0.821
Testing Random Forest  8.4s — acc=0.923 f1=0.901
Testing MLP...         6.7s — acc=0.891 f1=0.878

## 5. Main experiment loop

For each dataset:
- **Phase 1 (baseline):** 10-fold CV, train/evaluate all 5 classifiers on full feature set.
- **Phase 2 (ReliefF):** Same folds, but ReliefF is fit on the training fold only before classifiers are trained.

In [6]:
all_results = {}   # all_results[dataset][phase][classifier] = {acc, f1, params}
relieff_scores = {}  # feature importance scores per dataset

outer_cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

for ds_name, paths in DATASETS.items():
    print(f'\n=== Dataset: {ds_name} ===')
    X, y, le = load_dataset(paths['train'])

    all_results[ds_name] = {
        'baseline': {clf: {'acc': [], 'f1': [], 'params': []} for clf in CLASSIFIERS},
        'relieff':  {clf: {'acc': [], 'f1': [], 'params': []} for clf in CLASSIFIERS},
    }
    fold_feature_scores = []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):
        print(f'  Fold {fold_idx + 1}/{N_FOLDS}', end=' ... ')
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # ── Phase 1: baseline (full features) ────────────────────────────────
        for clf_name, (clf, grid) in CLASSIFIERS.items():
            acc, f1, params = evaluate_classifier(
                clf, grid, X_train, y_train, X_test, y_test)
            all_results[ds_name]['baseline'][clf_name]['acc'].append(acc)
            all_results[ds_name]['baseline'][clf_name]['f1'].append(f1)
            all_results[ds_name]['baseline'][clf_name]['params'].append(params)

        # ── Phase 2: ReliefF — fit on training fold ONLY ─────────────────────
        # Choose k via a quick inner CV sweep on the training fold
        best_k, best_k_score = K_CANDIDATES[0], -np.inf
        for k_candidate in K_CANDIDATES:
            if k_candidate >= X_train.shape[1]:
                continue
            Xtr_k, Xte_k, _, _ = apply_relieff(
                X_train, y_train, X_test, k=k_candidate)
            # quick inner CV with Random Forest as proxy
            inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)
            scores = []
            for itr, ite in inner.split(Xtr_k, y_train):
                rf = RandomForestClassifier(n_estimators=50,
                                            random_state=RANDOM_STATE)
                rf.fit(Xtr_k[itr], y_train[itr])
                scores.append(f1_score(y_train[ite],
                                        rf.predict(Xtr_k[ite]),
                                        average='macro', zero_division=0))
            if np.mean(scores) > best_k_score:
                best_k_score = np.mean(scores)
                best_k = k_candidate

        X_train_r, X_test_r, feat_scores, _ = apply_relieff(
            X_train, y_train, X_test, k=best_k)
        fold_feature_scores.append(feat_scores)

        for clf_name, (clf, grid) in CLASSIFIERS.items():
            acc, f1, params = evaluate_classifier(
                clf, grid, X_train_r, y_train, X_test_r, y_test)
            all_results[ds_name]['relieff'][clf_name]['acc'].append(acc)
            all_results[ds_name]['relieff'][clf_name]['f1'].append(f1)
            all_results[ds_name]['relieff'][clf_name]['params'].append(params)

        print('done.')

    # Average ReliefF scores across folds for plotting
    relieff_scores[ds_name] = np.mean(fold_feature_scores, axis=0)

print('\nAll experiments complete.')


=== Dataset: Dataset1 ===
  Loaded: 9583 samples, 19 features, 4 classes
  Fold 1/10 ... 

KeyboardInterrupt: 

## 6. Results summary table

In [ ]:
rows = []
for ds_name, phases in all_results.items():
    for phase_name, classifiers in phases.items():
        for clf_name, metrics in classifiers.items():
            rows.append({
                'Dataset':    ds_name,
                'Phase':      'Baseline' if phase_name == 'baseline' else 'ReliefF',
                'Classifier': clf_name,
                'Accuracy':   f"{np.mean(metrics['acc']):.4f} ± {np.std(metrics['acc']):.4f}",
                'Macro-F1':   f"{np.mean(metrics['f1']):.4f} ± {np.std(metrics['f1']):.4f}",
                'Best params (last fold)': metrics['params'][-1],
            })

results_df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_rows', 200)
display(results_df)

In [ ]:
# Save to CSV for the report
results_df.to_csv('results_summary.csv', index=False)
print('Saved to results_summary.csv')

## 7. Visualisation — Phase 1 vs Phase 2 comparison

In [ ]:
def plot_comparison(ds_name, metric='f1'):
    clf_names = list(CLASSIFIERS.keys())
    baseline_means = [np.mean(all_results[ds_name]['baseline'][c][metric])
                      for c in clf_names]
    baseline_stds  = [np.std(all_results[ds_name]['baseline'][c][metric])
                      for c in clf_names]
    relieff_means  = [np.mean(all_results[ds_name]['relieff'][c][metric])
                      for c in clf_names]
    relieff_stds   = [np.std(all_results[ds_name]['relieff'][c][metric])
                      for c in clf_names]

    x = np.arange(len(clf_names))
    width = 0.35
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - width/2, baseline_means, width, yerr=baseline_stds,
           label='Baseline (all features)', capsize=4, color='steelblue')
    ax.bar(x + width/2, relieff_means,  width, yerr=relieff_stds,
           label='ReliefF selected', capsize=4, color='coral')
    ax.set_xticks(x)
    ax.set_xticklabels(clf_names, rotation=15, ha='right')
    ax.set_ylabel('Macro-F1' if metric == 'f1' else 'Accuracy')
    ax.set_ylim(0, 1.05)
    ax.set_title(f'{ds_name} — Baseline vs ReliefF ({"Macro-F1" if metric == "f1" else "Accuracy"})')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{ds_name}_{metric}_comparison.png', dpi=150)
    plt.show()

for ds_name in DATASETS:
    plot_comparison(ds_name, metric='f1')
    plot_comparison(ds_name, metric='acc')

## 8. ReliefF feature importance — top features per dataset

In [ ]:
def plot_feature_importance(ds_name, X, top_n=30):
    scores = relieff_scores[ds_name]
    feature_names = [f'F{i}' for i in range(len(scores))]
    top_idx = np.argsort(scores)[::-1][:top_n]
    top_scores = scores[top_idx]
    top_names  = [feature_names[i] for i in top_idx]

    fig, ax = plt.subplots(figsize=(12, 5))
    colors = ['coral' if s > 0 else 'steelblue' for s in top_scores]
    ax.bar(range(top_n), top_scores, color=colors)
    ax.set_xticks(range(top_n))
    ax.set_xticklabels(top_names, rotation=90, fontsize=8)
    ax.set_ylabel('ReliefF score (mean across folds)')
    ax.set_title(f'{ds_name} — Top {top_n} features by ReliefF score')
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{ds_name}_relieff_importance.png', dpi=150)
    plt.show()

for ds_name, paths in DATASETS.items():
    X, y, _ = load_dataset(paths['train'])
    plot_feature_importance(ds_name, X, top_n=min(30, X.shape[1]))

## 9. Best hyperparameters summary

In [ ]:
from collections import Counter

print('Most common best hyperparameters per classifier per phase:\n')
for ds_name in DATASETS:
    print(f'── {ds_name} ──')
    for phase in ['baseline', 'relieff']:
        print(f'  Phase: {phase}')
        for clf_name in CLASSIFIERS:
            param_list = all_results[ds_name][phase][clf_name]['params']
            # Flatten to key:value strings and count most common
            param_strs = [str(sorted(p.items())) for p in param_list]
            most_common = Counter(param_strs).most_common(1)[0][0]
            print(f'    {clf_name}: {most_common}')
    print()

## 10. Discussion notes

Fill these in after running the experiments:

**Effect of ReliefF on performance:**
- Which classifiers improved most after feature selection?
- Which datasets benefited most from dimensionality reduction?
- Were there cases where ReliefF hurt performance? Why might that happen?

**ReliefF behaviour:**
- How many features were selected (best k) for each dataset?
- Were the top-ranked features consistent across folds?
- What does a negative ReliefF score for a feature mean?

**Computational cost:**
- ReliefF scales as O(n × m × k_neighbors) — note any runtime differences between datasets.

---
*End of notebook*